In [3]:
import duckdb

In [4]:
import os
os.listdir('D:/akiml_data/vitaldb')

['.ipynb_checkpoints',
 'clinical_data.csv',
 'clinical_parameters.csv',
 'lab_data.csv',
 'lab_parameters.csv',
 'track_names.csv',
 'Untitled.ipynb']

In [5]:
duckdb.sql("SELECT COUNT(*) AS n FROM 'D:/akiml_data/vitaldb/clinical_data.csv'").df()

,n
0,6388


In [6]:
duckdb.sql("SELECT * FROM 'D:/akiml_data/vitaldb/lab_parameters.csv'").df()

,Parameter,Category,Description,Unit,Reference value
0,wbc,CBC,White blood cell count,×1000/mcL,4~10
1,hb,CBC,Hemoglobin,g/dL,13~17
2,hct,CBC,Hematocrit,%,39~52
3,plt,CBC,Platelet count,×1000/mcL,130~400
4,esr,CBC,Erythrocyte sedimentation rate,mm/hr,0~9
5,gluc,Chemistry,Glucose,mg/dL,70~110
6,tprot,Chemistry,Total protein,g/dL,6.0~8.0
7,alb,Chemistry,Albumin,g/dL,3.3~5.2
8,tbil,Chemistry,Total bilirubin,mg/dL,0.2~1.2
9,ast,Chemistry,Asparate transferase,IU/L,1~40


In [7]:
duckdb.sql("SELECT * FROM 'D:/akiml_data/vitaldb/lab_data.csv' LIMIT 5").df()

,caseid,dt,name,result
0,1,594470,alb,2.9
1,1,399575,alb,3.2
2,1,12614,alb,3.4
3,1,137855,alb,3.6
4,1,399575,alt,12.0


In [8]:
duckdb.sql("SELECT COUNT(*) AS n_cre FROM 'D:/akiml_data/vitaldb/lab_data.csv' WHERE name = 'CRE'").df()

,n_cre
0,0


In [9]:
duckdb.sql("SELECT * FROM 'D:/akiml_data/vitaldb/clinical_data.csv' LIMIT 3").df()

,caseid,subjectid,casestart,caseend,anestart,aneend,opstart,opend,adm,dis,...,intraop_colloid,intraop_ppf,intraop_mdz,intraop_ftn,intraop_rocu,intraop_vecu,intraop_eph,intraop_phe,intraop_epi,intraop_ca
0,1,5955,0,11542,-552,10848.0,1668,10368,-236220,627780,...,0,120,0.0,100,70,0,10,0,0,0
1,2,2487,0,15741,-1039,14921.0,1721,14621,-221160,1506840,...,0,150,0.0,0,100,0,20,0,0,0
2,3,2861,0,4394,-590,4210.0,1090,3010,-218640,40560,...,0,0,0.0,0,50,0,0,0,0,0


In [10]:
duckdb.sql("SELECT DISTINCT name FROM 'D:/akiml_data/vitaldb/lab_data.csv' ORDER BY name").df()

,name
0,alb
1,alt
2,ammo
3,aptt
4,ast
5,be
6,bun
7,ccr
8,cl
9,cr


In [11]:
duckdb.sql("""
SELECT COUNT(*) AS n, MIN(result) AS min_val, MAX(result) AS max_val, AVG(result) AS mean_val
FROM 'D:/akiml_data/vitaldb/lab_data.csv'
WHERE name = 'cre'
""").df()

,n,min_val,max_val,mean_val
0,0,NaN,NaN,NaN


In [12]:
duckdb.sql("""
SELECT COUNT(*) AS n, MIN(result) AS min_val, MAX(result) AS max_val, AVG(result) AS mean_val
FROM 'D:/akiml_data/vitaldb/lab_data.csv'
WHERE name = 'cr'
""").df()

,n,min_val,max_val,mean_val
0,37311,0.02,64.88,1.314156


In [13]:
duckdb.sql("""
SELECT l.caseid, l.dt, l.result, c.opstart, c.opend,
       (l.dt - c.opstart)/3600.0 AS hours_from_opstart
FROM 'D:/akiml_data/vitaldb/lab_data.csv' l
JOIN 'D:/akiml_data/vitaldb/clinical_data.csv' c ON l.caseid = c.caseid
WHERE l.name = 'cr'
LIMIT 10
""").df()

,caseid,dt,result,opstart,opend,hours_from_opstart
0,2700,-654863,1.00,4611,26511,-183.187222
1,2700,-664770,1.02,4611,26511,-185.939167
2,2700,-633014,1.11,4611,26511,-177.118056
3,2700,-633072,1.14,4611,26511,-177.134167
4,2700,-613994,1.16,4611,26511,-171.834722
5,2700,-742610,1.22,4611,26511,-207.561389
6,2700,-679392,1.22,4611,26511,-190.000833
7,2700,-776708,1.27,4611,26511,-217.033056
8,2700,-599672,1.27,4611,26511,-167.856389
9,2700,-585716,1.33,4611,26511,-163.979722


In [14]:
duckdb.sql("""
WITH cr AS (
  SELECT l.caseid, l.dt, l.result AS cr_val, c.opstart, c.opend
  FROM 'D:/akiml_data/vitaldb/lab_data.csv' l
  JOIN 'D:/akiml_data/vitaldb/clinical_data.csv' c ON l.caseid = c.caseid
  WHERE l.name = 'cr' AND l.result BETWEEN 0.2 AND 25
),
base AS (
  SELECT caseid, cr_val AS baseline_cr
  FROM (
    SELECT caseid, dt, cr_val,
           ROW_NUMBER() OVER (PARTITION BY caseid ORDER BY dt DESC) AS rn
    FROM cr
    WHERE dt <= opstart AND dt >= opstart - 7*86400
  ) WHERE rn = 1
),
post AS (
  SELECT caseid, MAX(cr_val) AS postop_cr_max
  FROM cr
  WHERE dt >= opend AND dt <= opend + 7*86400
  GROUP BY caseid
)
SELECT
  COUNT(*) AS n_with_baseline,
  COUNT(post.caseid) AS n_with_postop_cr,
  SUM(CASE WHEN postop_cr_max >= GREATEST(baseline_cr*1.5, baseline_cr+0.3) THEN 1 ELSE 0 END) AS n_aki
FROM base LEFT JOIN post USING (caseid)
""").df()

,n_with_baseline,n_with_postop_cr,n_aki
0,3741,3103,105.0


In [15]:
duckdb.sql("""
SELECT
  COUNT(*) AS n,
  ROUND(MEDIAN(baseline_cr),2) AS median_base,
  ROUND(QUANTILE_CONT(baseline_cr, 0.05),2) AS p05,
  ROUND(QUANTILE_CONT(baseline_cr, 0.25),2) AS p25
FROM (
  SELECT l.caseid, l.result AS baseline_cr,
         ROW_NUMBER() OVER (PARTITION BY l.caseid ORDER BY l.dt DESC) AS rn
  FROM 'D:/akiml_data/vitaldb/lab_data.csv' l
  JOIN 'D:/akiml_data/vitaldb/clinical_data.csv' c ON l.caseid = c.caseid
  WHERE l.name='cr' AND l.result BETWEEN 0.2 AND 25
    AND l.dt <= c.opstart AND l.dt >= c.opstart - 7*86400
) WHERE rn = 1
""").df()

InternalException: INTERNAL Error: Failed to cast expression to type - expression type mismatch
This error signals an assertion failure within DuckDB. This usually occurs due to unexpected conditions or errors in the program's logic.
For more information, see https://duckdb.org/docs/stable/dev/internal_errors

In [16]:
duckdb.sql("""
WITH cr AS (
  SELECT l.caseid, l.dt, l.result AS cr_val, c.opstart, c.opend
  FROM 'D:/akiml_data/vitaldb/lab_data.csv' l
  JOIN 'D:/akiml_data/vitaldb/clinical_data.csv' c ON l.caseid = c.caseid
  WHERE l.name='cr' AND l.result BETWEEN 0.2 AND 25
),
base AS (
  SELECT caseid, cr_val AS baseline_cr
  FROM (SELECT caseid, dt, cr_val,
        ROW_NUMBER() OVER (PARTITION BY caseid ORDER BY dt DESC) rn
        FROM cr WHERE dt <= opstart AND dt >= opstart - 7*86400)
  WHERE rn = 1
),
flag AS (
  SELECT cr.caseid,
    MAX(CASE WHEN cr.dt >= cr.opend AND cr.dt <= cr.opend + 2*86400
              AND cr_val >= base.baseline_cr + 0.3 THEN 1 ELSE 0 END) AS crit_48h,
    MAX(CASE WHEN cr.dt >= cr.opend AND cr.dt <= cr.opend + 7*86400
              AND cr_val >= base.baseline_cr * 1.5 THEN 1 ELSE 0 END) AS crit_7d
  FROM cr JOIN base ON cr.caseid = base.caseid
  GROUP BY cr.caseid
)
SELECT
  COUNT(*) AS n_total,
  SUM(crit_48h) AS n_aki_48h,
  SUM(crit_7d) AS n_aki_7d,
  SUM(CASE WHEN crit_48h=1 OR crit_7d=1 THEN 1 ELSE 0 END) AS n_aki_kdigo
FROM base LEFT JOIN flag USING (caseid)
""").df()

,n_total,n_aki_48h,n_aki_7d,n_aki_kdigo
0,3741,163.0,115.0,195.0


In [17]:
duckdb.sql("""
SELECT CASE WHEN baseline_cr < 0.5 THEN 'a.<0.5'
            WHEN baseline_cr < 0.7 THEN 'b.0.5-0.7'
            WHEN baseline_cr < 1.0 THEN 'c.0.7-1.0'
            WHEN baseline_cr < 1.5 THEN 'd.1.0-1.5'
            ELSE 'e.>=1.5' END AS base_grp,
       COUNT(*) AS n
FROM (
  SELECT l.caseid, l.result AS baseline_cr,
         ROW_NUMBER() OVER (PARTITION BY l.caseid ORDER BY l.dt DESC) AS rn
  FROM 'D:/akiml_data/vitaldb/lab_data.csv' l
  JOIN 'D:/akiml_data/vitaldb/clinical_data.csv' c ON l.caseid = c.caseid
  WHERE l.name='cr' AND l.result BETWEEN 0.2 AND 25
    AND l.dt <= c.opstart AND l.dt >= c.opstart - 7*86400
) WHERE rn = 1
GROUP BY 1 ORDER BY 1
""").df()

InternalException: INTERNAL Error: Failed to cast expression to type - expression type mismatch
This error signals an assertion failure within DuckDB. This usually occurs due to unexpected conditions or errors in the program's logic.
For more information, see https://duckdb.org/docs/stable/dev/internal_errors

In [18]:
base = duckdb.sql("""
WITH cr AS (
  SELECT l.caseid, l.dt, l.result AS cr_val, c.opstart
  FROM 'D:/akiml_data/vitaldb/lab_data.csv' l
  JOIN 'D:/akiml_data/vitaldb/clinical_data.csv' c ON l.caseid = c.caseid
  WHERE l.name='cr' AND l.result BETWEEN 0.2 AND 25
)
SELECT caseid, cr_val AS baseline_cr
FROM (
  SELECT caseid, dt, cr_val,
         ROW_NUMBER() OVER (PARTITION BY caseid ORDER BY dt DESC) AS rn
  FROM cr
  WHERE dt <= opstart AND dt >= opstart - 7*86400
) WHERE rn = 1
""").df()

InternalException: INTERNAL Error: Failed to cast expression to type - expression type mismatch
This error signals an assertion failure within DuckDB. This usually occurs due to unexpected conditions or errors in the program's logic.
For more information, see https://duckdb.org/docs/stable/dev/internal_errors